In [1]:
import pandas as pd
import numpy as np
import pickle
import shap

with open("data/credit_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("data/feature_list.pkl", "rb") as f:
    features = pickle.load(f)

df = pd.read_csv("data/application_train_features.csv")
X = df[features]

explainer = shap.TreeExplainer(model)
print("✅ Ready")

✅ Ready


In [2]:
# Maps each feature name to a plain English explanation
# Format: (positive_reason, negative_reason)
# positive = this feature is HURTING the applicant (pushing toward default)
# negative = this feature is HELPING the applicant (pushing away from default)

reason_map = {
    "EXT_SOURCE_1": (
        "External credit score 1 is low, indicating poor credit history",
        "External credit score 1 is strong, indicating good credit history"
    ),
    "EXT_SOURCE_2": (
        "External credit score 2 is low, indicating poor credit history",
        "External credit score 2 is strong, indicating good credit history"
    ),
    "EXT_SOURCE_3": (
        "External credit score 3 is low, indicating poor credit history",
        "External credit score 3 is strong, indicating good credit history"
    ),
    "CREDIT_TO_GOODS": (
        "Loan amount significantly exceeds the goods price — overborrowing detected",
        "Loan amount is well matched to the goods price"
    ),
    "ANNUITY_TO_INCOME": (
        "Monthly repayment is high relative to income — repayment burden is heavy",
        "Monthly repayment is manageable relative to income"
    ),
    "CREDIT_TO_INCOME": (
        "Total loan is very large relative to annual income",
        "Total loan is reasonable relative to annual income"
    ),
    "EMPLOYMENT_RATIO": (
        "Limited employment history relative to age",
        "Strong employment history relative to age"
    ),
    "DAYS_EMPLOYED": (
        "Short employment duration at current job",
        "Long employment duration — stable job history"
    ),
    "DAYS_BIRTH": (
        "Applicant age is a risk factor for this loan profile",
        "Applicant age is a positive factor for this loan profile"
    ),
    "AMT_CREDIT": (
        "Loan amount is high",
        "Loan amount is within acceptable range"
    ),
    "AMT_INCOME_TOTAL": (
        "Income level is low relative to loan obligations",
        "Income level is strong relative to loan obligations"
    ),
    "INCOME_PER_PERSON": (
        "Income per family member is low — household financial pressure is high",
        "Income per family member is healthy"
    ),
    "DOCUMENT_COUNT": (
        "Few documents provided — identity and income harder to verify",
        "Good documentation provided — identity and income well verified"
    ),
    "NAME_EDUCATION_TYPE_ENC": (
        "Education level is associated with higher default rates",
        "Education level is associated with lower default rates"
    ),
    "EXT_SOURCE_1_MISSING": (
        "No external credit score 1 available — limited credit history",
        "External credit score 1 is available"
    ),
}

print(f"✅ Reason map built — {len(reason_map)} features covered")

✅ Reason map built — 15 features covered


In [3]:
def generate_reason_codes(applicant_id, top_n=3):
    """
    Takes an applicant ID, computes SHAP values,
    returns plain English reasons for the credit decision.
    """
    # Get applicant data
    applicant = X.loc[[applicant_id]]
    
    # Compute SHAP values
    shap_vals = explainer.shap_values(applicant)[0]
    
    # Get risk score
    risk_score = model.predict_proba(applicant)[:, 1][0]
    decision = "DECLINE" if risk_score > 0.5 else "APPROVE"
    
    # Pair features with their SHAP values
    shap_df = pd.DataFrame({
        "feature": features,
        "shap_value": shap_vals,
        "feature_value": applicant.values[0]
    }).sort_values("shap_value", key=abs, ascending=False)
    
    # Generate reasons
    risk_factors = []      # pushing toward default (positive SHAP)
    protective_factors = [] # pushing away from default (negative SHAP)
    
    for _, row in shap_df.head(10).iterrows():
        feat = row["feature"]
        if feat not in reason_map:
            continue
        pos_reason, neg_reason = reason_map[feat]
        if row["shap_value"] > 0:
            risk_factors.append(f"⚠️  {pos_reason}")
        else:
            protective_factors.append(f"✅  {neg_reason}")
    
    # Print structured report
    print("=" * 55)
    print(f"CREDIT DECISION REPORT — Applicant {applicant_id}")
    print("=" * 55)
    print(f"Risk Score:  {risk_score:.3f}  (0=safe, 1=high risk)")
    print(f"Decision:    {decision}")
    print()
    print("Top Risk Factors:")
    for r in risk_factors[:top_n]:
        print(f"  {r}")
    print()
    print("Protective Factors:")
    for r in protective_factors[:top_n]:
        print(f"  {r}")
    print("=" * 55)
    
    return risk_score, decision, risk_factors, protective_factors

print("✅ Function ready")

✅ Function ready


In [4]:
# Use the same defaulter from Step 10
defaulter_idx = df[df["TARGET"] == 1].sample(1, random_state=42).index[0]
generate_reason_codes(defaulter_idx)

CREDIT DECISION REPORT — Applicant 71233
Risk Score:  0.782  (0=safe, 1=high risk)
Decision:    DECLINE

Top Risk Factors:
  ⚠️  External credit score 2 is low, indicating poor credit history
  ⚠️  External credit score 1 is low, indicating poor credit history
  ⚠️  Loan amount is high

Protective Factors:
  ✅  Long employment duration — stable job history
  ✅  External credit score 1 is available
  ✅  Strong employment history relative to age


(0.7823171,
 'DECLINE',
 ['⚠️  External credit score 2 is low, indicating poor credit history',
  '⚠️  External credit score 1 is low, indicating poor credit history',
  '⚠️  Loan amount is high',
  '⚠️  Monthly repayment is high relative to income — repayment burden is heavy',
  '⚠️  Loan amount significantly exceeds the goods price — overborrowing detected',
  '⚠️  Applicant age is a risk factor for this loan profile'],
 ['✅  Long employment duration — stable job history',
  '✅  External credit score 1 is available',
  '✅  Strong employment history relative to age'])

In [5]:
safe_idx = df[df["TARGET"] == 0].sample(1, random_state=42).index[0]
generate_reason_codes(safe_idx)

CREDIT DECISION REPORT — Applicant 201622
Risk Score:  0.311  (0=safe, 1=high risk)
Decision:    APPROVE

Top Risk Factors:
  ⚠️  Applicant age is a risk factor for this loan profile
  ⚠️  No external credit score 1 available — limited credit history
  ⚠️  Short employment duration at current job

Protective Factors:
  ✅  External credit score 3 is strong, indicating good credit history
  ✅  External credit score 2 is strong, indicating good credit history
  ✅  Loan amount is within acceptable range


(0.311366,
 'APPROVE',
 ['⚠️  Applicant age is a risk factor for this loan profile',
  '⚠️  No external credit score 1 available — limited credit history',
  '⚠️  Short employment duration at current job',
  '⚠️  Education level is associated with higher default rates'],
 ['✅  External credit score 3 is strong, indicating good credit history',
  '✅  External credit score 2 is strong, indicating good credit history',
  '✅  Loan amount is within acceptable range',
  '✅  Loan amount is well matched to the goods price',
  '✅  Monthly repayment is manageable relative to income'])

In [6]:
def generate_counterfactual(applicant_id, threshold=0.5):
    """
    For a DECLINED applicant, finds the single most impactful
    thing they could improve to move toward approval.
    """
    applicant = X.loc[[applicant_id]]
    risk_score = model.predict_proba(applicant)[:, 1][0]
    
    if risk_score <= threshold:
        print(f"Applicant {applicant_id} is already APPROVED (score: {risk_score:.3f})")
        return
    
    # Get SHAP values
    shap_vals = explainer.shap_values(applicant)[0]
    
    # Find features pushing toward default (positive SHAP)
    shap_df = pd.DataFrame({
        "feature": features,
        "shap_value": shap_vals,
        "feature_value": applicant.values[0]
    })
    
    # Only actionable features — bureau scores aren't instantly changeable
    actionable = [
        "ANNUITY_TO_INCOME",
        "CREDIT_TO_GOODS", 
        "CREDIT_TO_INCOME",
        "EMPLOYMENT_RATIO",
        "DAYS_EMPLOYED",
        "DOCUMENT_COUNT",
        "INCOME_PER_PERSON",
    ]
    
    actionable_shap = shap_df[
        (shap_df["feature"].isin(actionable)) &
        (shap_df["shap_value"] > 0)  # only features hurting the applicant
    ].sort_values("shap_value", ascending=False)
    
    print("=" * 55)
    print(f"COUNTERFACTUAL REPORT — Applicant {applicant_id}")
    print("=" * 55)
    print(f"Current Risk Score: {risk_score:.3f} — DECLINED")
    print(f"Approval Threshold: {threshold:.2f}")
    print(f"Score needs to drop by: {risk_score - threshold:.3f}")
    print()
    print("What could improve this decision:")
    print()
    
    counterfactual_map = {
        "ANNUITY_TO_INCOME": lambda v: (
            f"Monthly repayment burden is {v:.1%} of income.\n"
            f"   → Requesting a smaller loan or longer tenure would reduce\n"
            f"     this ratio and lower risk score."
        ),
        "CREDIT_TO_GOODS": lambda v: (
            f"Loan amount is {v:.2f}x the goods price — overborrowing detected.\n"
            f"   → Aligning loan amount closer to goods value (ratio near 1.0)\n"
            f"     would significantly reduce risk score."
        ),
        "CREDIT_TO_INCOME": lambda v: (
            f"Total loan is {v:.1f}x annual income — very high leverage.\n"
            f"   → A smaller loan amount relative to income would help."
        ),
        "EMPLOYMENT_RATIO": lambda v: (
            f"Employment covers {v:.1%} of working life — limited stability signal.\n"
            f"   → Longer tenure at current employer would strengthen this."
        ),
        "DAYS_EMPLOYED": lambda v: (
            f"Current job tenure is {abs(v)/365:.1f} years.\n"
            f"   → Applying after building longer employment history would help."
        ),
        "DOCUMENT_COUNT": lambda v: (
            f"Only {int(v)} documents provided.\n"
            f"   → Submitting additional identity and income documents\n"
            f"     would improve verifiability."
        ),
        "INCOME_PER_PERSON": lambda v: (
            f"Income per family member is ₹{v:,.0f}/month.\n"
            f"   → Demonstrating additional income sources would strengthen\n"
            f"     this profile."
        ),
    }
    
    shown = 0
    for _, row in actionable_shap.iterrows():
        feat = row["feature"]
        val = row["feature_value"]
        shap_impact = row["shap_value"]
        
        if feat in counterfactual_map and shown < 2:
            print(f"  #{shown+1} — {counterfactual_map[feat](val)}")
            print(f"       (Risk impact: +{shap_impact:.3f})")
            print()
            shown += 1
    
    if shown == 0:
        print("  Primary risk drivers are bureau scores —")
        print("  improving repayment history over 6-12 months")
        print("  is the most impactful path to approval.")
    
    print("=" * 55)

print("✅ Counterfactual generator ready")

✅ Counterfactual generator ready


In [7]:
generate_counterfactual(defaulter_idx)

COUNTERFACTUAL REPORT — Applicant 71233
Current Risk Score: 0.782 — DECLINED
Approval Threshold: 0.50
Score needs to drop by: 0.282

What could improve this decision:

  #1 — Monthly repayment burden is 22.7% of income.
   → Requesting a smaller loan or longer tenure would reduce
     this ratio and lower risk score.
       (Risk impact: +0.094)

  #2 — Loan amount is 1.21x the goods price — overborrowing detected.
   → Aligning loan amount closer to goods value (ratio near 1.0)
     would significantly reduce risk score.
       (Risk impact: +0.072)



In [8]:
# Find someone just above the threshold — most actionable scenario
df_val = df.copy()
df_val["risk_score"] = model.predict_proba(X)[:, 1]

# Borderline = score between 0.50 and 0.65
borderline = df_val[
    (df_val["risk_score"] > 0.50) & 
    (df_val["risk_score"] < 0.65)
].sample(1, random_state=7)

borderline_idx = borderline.index[0]
print(f"Borderline applicant score: {borderline['risk_score'].values[0]:.3f}")
generate_counterfactual(borderline_idx)

Borderline applicant score: 0.518
COUNTERFACTUAL REPORT — Applicant 273217
Current Risk Score: 0.518 — DECLINED
Approval Threshold: 0.50
Score needs to drop by: 0.018

What could improve this decision:

  #1 — Loan amount is 1.16x the goods price — overborrowing detected.
   → Aligning loan amount closer to goods value (ratio near 1.0)
     would significantly reduce risk score.
       (Risk impact: +0.081)

  #2 — Current job tenure is 3.2 years.
   → Applying after building longer employment history would help.
       (Risk impact: +0.056)



In [9]:
def risk_to_credit_score(risk_probability):
    """
    Converts 0-1 risk probability to a 300-900 credit score.
    Higher score = lower risk (matches CIBIL convention)
    """
    score = round(900 - (risk_probability * 600))
    
    if score >= 750:
        band = "Excellent"
        color = "🟢"
    elif score >= 700:
        band = "Good"
        color = "🟡"
    elif score >= 650:
        band = "Fair"
        color = "🟠"
    elif score >= 600:
        band = "Poor"
        color = "🔴"
    else:
        band = "Very Poor"
        color = "🔴"
    
    return score, band, color

# Test it across the range
test_scores = [0.1, 0.3, 0.5, 0.518, 0.782, 0.9]
print(f"{'Risk Prob':<12} {'Credit Score':<14} {'Band':<12} {'Decision'}")
print("-" * 50)
for prob in test_scores:
    score, band, color = risk_to_credit_score(prob)
    decision = "APPROVE" if prob <= 0.5 else "DECLINE"
    print(f"{prob:<12.3f} {score:<14} {color} {band:<10} {decision}")

Risk Prob    Credit Score   Band         Decision
--------------------------------------------------
0.100        840            🟢 Excellent  APPROVE
0.300        720            🟡 Good       APPROVE
0.500        600            🔴 Poor       APPROVE
0.518        589            🔴 Very Poor  DECLINE
0.782        431            🔴 Very Poor  DECLINE
0.900        360            🔴 Very Poor  DECLINE


In [10]:
def full_decision_report(applicant_id, top_n=3):
    """
    Complete credit decision report combining:
    - Credit score (consumer-facing)
    - Risk probability (internal)
    - Reason codes (plain English)
    - Counterfactuals (actionable advice)
    """
    applicant = X.loc[[applicant_id]]
    risk_prob = model.predict_proba(applicant)[:, 1][0]
    credit_score, band, color = risk_to_credit_score(risk_prob)
    decision = "DECLINE" if risk_prob > 0.5 else "APPROVE"
    
    # SHAP values
    shap_vals = explainer.shap_values(applicant)[0]
    shap_df = pd.DataFrame({
        "feature": features,
        "shap_value": shap_vals,
        "feature_value": applicant.values[0]
    }).sort_values("shap_value", key=abs, ascending=False)
    
    # Reason codes
    risk_factors = []
    protective_factors = []
    for _, row in shap_df.head(10).iterrows():
        feat = row["feature"]
        if feat not in reason_map:
            continue
        pos, neg = reason_map[feat]
        if row["shap_value"] > 0:
            risk_factors.append(pos)
        else:
            protective_factors.append(neg)
    
    # Print loan officer view
    print("=" * 55)
    print(f"  CREDEXPLAIN DECISION REPORT")
    print(f"  Applicant ID: {applicant_id}")
    print("=" * 55)
    print()
    print(f"  LOAN OFFICER VIEW (internal)")
    print(f"  Risk Probability : {risk_prob:.3f}")
    print(f"  Decision         : {decision}")
    print()
    print(f"  APPLICANT VIEW (consumer-facing)")
    print(f"  Credit Score     : {credit_score}  {color} {band}")
    print(f"  Approval Status  : {decision}")
    print()
    print("  TOP RISK FACTORS")
    for i, r in enumerate(risk_factors[:top_n], 1):
        print(f"  {i}. ⚠️  {r}")
    print()
    print("  PROTECTIVE FACTORS")
    for i, r in enumerate(protective_factors[:top_n], 1):
        print(f"  {i}. ✅  {r}")
    
    # Counterfactuals only for declined
    if decision == "DECLINE":
        print()
        print("  WHAT COULD CHANGE THIS DECISION")
        actionable = [
            "ANNUITY_TO_INCOME", "CREDIT_TO_GOODS",
            "CREDIT_TO_INCOME", "DAYS_EMPLOYED",
            "DOCUMENT_COUNT", "INCOME_PER_PERSON"
        ]
        cf_map = {
            "ANNUITY_TO_INCOME": lambda v: f"Repayment burden is {v:.1%} of income → request smaller loan or longer tenure",
            "CREDIT_TO_GOODS":   lambda v: f"Loan is {v:.2f}x goods price → align loan closer to goods value",
            "CREDIT_TO_INCOME":  lambda v: f"Loan is {v:.1f}x annual income → reduce loan amount",
            "DAYS_EMPLOYED":     lambda v: f"Job tenure is {abs(v)/365:.1f} years → longer tenure strengthens profile",
            "DOCUMENT_COUNT":    lambda v: f"Only {int(v)} documents provided → submit additional income proof",
            "INCOME_PER_PERSON": lambda v: f"₹{v:,.0f}/month per family member → demonstrate additional income",
        }
        actionable_shap = shap_df[
            (shap_df["feature"].isin(actionable)) &
            (shap_df["shap_value"] > 0)
        ].head(2)
        
        shown = 0
        for _, row in actionable_shap.iterrows():
            feat = row["feature"]
            if feat in cf_map:
                print(f"  {shown+1}. 💡 {cf_map[feat](row['feature_value'])}")
                shown += 1
        if shown == 0:
            print("  💡 Build repayment history over 6-12 months")
    
    print()
    print("=" * 55)
    
    return {
        "applicant_id": applicant_id,
        "risk_probability": round(float(risk_prob), 4),
        "credit_score": credit_score,
        "band": band,
        "decision": decision,
        "risk_factors": risk_factors[:top_n],
        "protective_factors": protective_factors[:top_n],
    }

print("✅ Full decision report function ready")

✅ Full decision report function ready


In [11]:
report_high = full_decision_report(defaulter_idx)
print()
report_low = full_decision_report(safe_idx)

  CREDEXPLAIN DECISION REPORT
  Applicant ID: 71233

  LOAN OFFICER VIEW (internal)
  Risk Probability : 0.782
  Decision         : DECLINE

  APPLICANT VIEW (consumer-facing)
  Credit Score     : 431  🔴 Very Poor
  Approval Status  : DECLINE

  TOP RISK FACTORS
  1. ⚠️  External credit score 2 is low, indicating poor credit history
  2. ⚠️  External credit score 1 is low, indicating poor credit history
  3. ⚠️  Loan amount is high

  PROTECTIVE FACTORS
  1. ✅  Long employment duration — stable job history
  2. ✅  External credit score 1 is available
  3. ✅  Strong employment history relative to age

  WHAT COULD CHANGE THIS DECISION
  1. 💡 Repayment burden is 22.7% of income → request smaller loan or longer tenure
  2. 💡 Loan is 1.21x goods price → align loan closer to goods value


  CREDEXPLAIN DECISION REPORT
  Applicant ID: 201622

  LOAN OFFICER VIEW (internal)
  Risk Probability : 0.311
  Decision         : APPROVE

  APPLICANT VIEW (consumer-facing)
  Credit Score     : 713  🟡 